Before Silver, we need to prove:
What arrived in staging = what landed in Bronze

In [0]:
from pyspark.sql import functions as F

BASE_PATH = "/Volumes/customer360_dev/staging/source_files"

ENTITIES = [
    "customers",
    "addresses",
    "subscriptions",
    "products",
    "orders",
    "order_items",
    "payments",
    "support_tickets"
]

### Compare Staging vs Bronze

In [0]:
reconciliation_results = []

for entity in ENTITIES:

    staging_path = f"{BASE_PATH}/{entity}/initial"
    bronze_table = f"customer360_dev.bronze.{entity}"

    staging_df = (
        spark.read
        .option("header", True)
        .option("inferSchema", False)
        .csv(staging_path)
    )

    bronze_df = spark.table(bronze_table)

    staging_count = staging_df.count()
    bronze_count = bronze_df.count()

    difference = bronze_count - staging_count

    reconciliation_results.append({
        "entity": entity,
        "staging_count": staging_count,
        "bronze_count": bronze_count,
        "difference": difference,
        "status": "PASS" if difference == 0 else "FAIL"
    })

reconciliation_df = spark.createDataFrame(
    reconciliation_results
)

display(reconciliation_df)

bronze_count,difference,entity,staging_count,status
1000000,0,customers,1000000,PASS
1200078,0,addresses,1200078,PASS
1000066,0,subscriptions,1000066,PASS
150024,0,products,150024,PASS
5000126,0,orders,5000126,PASS
15000186,0,order_items,15000186,PASS
6000150,0,payments,6000150,PASS
1500096,0,support_tickets,1500096,PASS


### Validate Bronze metadata

In [0]:
metadata_results = []

for entity in ENTITIES:

    table_name = f"customer360_dev.bronze.{entity}"

    df = spark.table(table_name)

    required_metadata = [
        "_ingestion_timestamp",
        "_source_file",
        "_source_system",
        "_load_type"
    ]

    missing_columns = [
        col_name
        for col_name in required_metadata
        if col_name not in df.columns
    ]

    metadata_results.append({
        "entity": entity,
        "missing_metadata_columns": ",".join(missing_columns),
        "status": "PASS" if len(missing_columns) == 0 else "FAIL"
    })

display(
    spark.createDataFrame(metadata_results)
)

entity,missing_metadata_columns,status
customers,,PASS
addresses,,PASS
subscriptions,,PASS
products,,PASS
orders,,PASS
order_items,,PASS
payments,,PASS
support_tickets,,PASS


### Validate null metadata

In [0]:
metadata_null_results = []

for entity in ENTITIES:

    df = spark.table(
        f"customer360_dev.bronze.{entity}"
    )

    invalid_metadata_count = (
        df
        .filter(
            F.col("_ingestion_timestamp").isNull()
            | F.col("_source_file").isNull()
            | F.col("_source_system").isNull()
            | F.col("_load_type").isNull()
        )
        .count()
    )

    metadata_null_results.append({
        "entity": entity,
        "invalid_metadata_rows": invalid_metadata_count,
        "status": (
            "PASS"
            if invalid_metadata_count == 0
            else "FAIL"
        )
    })

display(
    spark.createDataFrame(metadata_null_results)
)

entity,invalid_metadata_rows,status
customers,0,PASS
addresses,0,PASS
subscriptions,0,PASS
products,0,PASS
orders,0,PASS
order_items,0,PASS
payments,0,PASS
support_tickets,0,PASS


### Validate load type

_load_type = INITIAL

In [0]:
for entity in ENTITIES:

    print(f"\n{entity.upper()}")

    (
        spark.table(
            f"customer360_dev.bronze.{entity}"
        )
        .groupBy("_load_type")
        .count()
        .show()
    )


CUSTOMERS
+----------+-------+
|_load_type|  count|
+----------+-------+
|   INITIAL|1000000|
+----------+-------+


ADDRESSES
+----------+-------+
|_load_type|  count|
+----------+-------+
|   INITIAL|1200078|
+----------+-------+


SUBSCRIPTIONS
+----------+-------+
|_load_type|  count|
+----------+-------+
|   INITIAL|1000066|
+----------+-------+


PRODUCTS
+----------+------+
|_load_type| count|
+----------+------+
|   INITIAL|150024|
+----------+------+


ORDERS
+----------+-------+
|_load_type|  count|
+----------+-------+
|   INITIAL|5000126|
+----------+-------+


ORDER_ITEMS
+----------+--------+
|_load_type|   count|
+----------+--------+
|   INITIAL|15000186|
+----------+--------+


PAYMENTS
+----------+-------+
|_load_type|  count|
+----------+-------+
|   INITIAL|6000150|
+----------+-------+


SUPPORT_TICKETS
+----------+-------+
|_load_type|  count|
+----------+-------+
|   INITIAL|1500096|
+----------+-------+



Validate source-file lineage

In [0]:
lineage_results = []

for entity in ENTITIES:

    df = spark.table(
        f"customer360_dev.bronze.{entity}"
    )

    file_count = (
        df
        .select("_source_file")
        .distinct()
        .count()
    )

    lineage_results.append({
        "entity": entity,
        "source_files_in_bronze": file_count
    })

display(
    spark.createDataFrame(lineage_results)
)

entity,source_files_in_bronze
customers,10
addresses,13
subscriptions,11
products,4
orders,21
order_items,31
payments,25
support_tickets,16


In [0]:
%sql
DESCRIBE HISTORY customer360_dev.bronze.orders;

version,timestamp,userId,userName,operation,operationParameters,job,notebook,queryHistoryStatementId,clusterId,readVersion,isolationLevel,isBlindAppend,operationMetrics,userMetadata,engineInfo
0,2026-09-26T09:59:55.000Z,75225367252892,pavankumar.m1432@gmail.com,CREATE OR REPLACE TABLE AS SELECT,"Map(isV1SaveAsTableOverwrite -> true, partitionBy -> [], clusterBy -> [], description -> null, isManaged -> true, properties -> {""delta.parquet.compression.codec"":""zstd"",""delta.parquet.format.version.afe.internal"":""2.12.0"",""delta.enableDeletionVectors"":""true"",""delta.parquet.format.version"":""2.12.0"",""io.unitycatalog.tableId"":""4939ff79-3e91-4bc3-aa4c-d86efbb0cf26""}, statsOnLoad -> true)",null,List(4147292987036836),1169ad90-0ffb-4a64-9b14-4b40e390e7c6,0926-084451-pieaaldy-v2n,null,WriteSerializable,false,"Map(numFiles -> 10, numRemovedFiles -> 0, numRemovedBytes -> 0, numDeletionVectorsRemoved -> 0, numOutputRows -> 5000126, numOutputBytes -> 38736918)",null,Databricks-Runtime/19.6.x-aarch64-photon-scala2.13
